In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm


def seed_everything(seed: int = 42):
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + ".png")
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [3]:
# Keep eval/inference transform identical to preserve evaluation semantics during prediction.
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# Minimal, standard augmentation for training only (helps generalization; same model/loss).
train_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(
            brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02
        ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [4]:
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"
test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=transform, test=True
)
test_loader = DataLoader(
    test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True
)



In [5]:
"""
model_paths = {
    'resnet18': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
    'efficientnet_b5': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/efficientnet_b5.pth",
    'inception_resnet_v2': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_resnet_v2.pth",
    'inception_v4': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_v4.pth",
    'seresnext50_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
    'seresnext101_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth"
}
"""
model_paths = {
    "resnet18": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
}

model_names = {
    "resnet18": "resnet18",
    "efficientnet_b5": "efficientnet_b5",
    "inception_resnet_v2": "inception_resnet_v2",
    "inception_v4": "inception_v4",
    "seresnext50_32x4d": "seresnext50_32x4d",
    "seresnext101_32x4d": "seresnext101_32x4d",
}



In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_list = []
missing_weights = []

for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=False, num_classes=5)
    if os.path.exists(path):
        state = torch.load(path, map_location="cpu")
        model.load_state_dict(state)
        model.to(device)
        model.eval()
        models_list.append(model)
    else:
        missing_weights.append((model_key, path))

print(
    f"Loaded {len(models_list)} model(s). Missing weights for {len(missing_weights)} model(s)."
)
if missing_weights:
    for mk, mp in missing_weights:
        print(f"Missing: {mk} -> {mp}")

if len(models_list) == 0:
    train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
    train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"

    # Use augmentation only for training to boost score while keeping the same model/loss.
    train_dataset = BlindnessDataset(
        train_csv_file, train_root_dir, transform=train_transform, test=False
    )
    train_loader = DataLoader(
        train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
    )

    model = timm.create_model("resnet18", pretrained=True, num_classes=5)
    model.to(device)

    # Class imbalance is strong in APTOS; weighting CE is a minimal change that usually improves kappa.
    train_df = pd.read_csv(train_csv_file)
    counts = (
        train_df["diagnosis"]
        .value_counts()
        .reindex([0, 1, 2, 3, 4], fill_value=0)
        .values.astype(np.float64)
    )
    counts = np.maximum(counts, 1.0)
    inv = 1.0 / counts
    class_weights = inv / inv.sum() * len(inv)
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

    # Keep Adam but slightly higher LR + gentle schedule; still same training approach (supervised CE).
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    # Small schedule improves convergence without changing core logic.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=3, eta_min=5e-5
    )

    model.train()
    epochs = 3  # minimal increase from 1 epoch to improve score toward target
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in tqdm(
            train_loader, desc=f"Training fallback model (epoch {epoch+1}/{epochs})"
        ):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item())

        scheduler.step()
        print(
            f"Epoch {epoch+1} loss: {running_loss / max(1, len(train_loader)):.4f} | lr: {scheduler.get_last_lr()[0]:.6f}"
        )

    model.eval()
    models_list = [model]



Loaded 0 model(s). Missing weights for 1 model(s).
Missing: resnet18 -> /kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Training fallback model (epoch 1/3): 100%|██████████| 206/206 [02:24<00:00,  1.43it/s]


Epoch 1 loss: 1.3098 | lr: 0.000237


Training fallback model (epoch 2/3): 100%|██████████| 206/206 [02:30<00:00,  1.37it/s]


Epoch 2 loss: 1.0391 | lr: 0.000113


Training fallback model (epoch 3/3): 100%|██████████| 206/206 [02:29<00:00,  1.38it/s]

Epoch 3 loss: 0.9146 | lr: 0.000050


In [7]:
all_outputs = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        outputs = [m(images).unsqueeze(0) for m in models_list]
        outputs = torch.cat(outputs, dim=0)
        averaged_outputs = torch.mean(outputs, dim=0)
        all_outputs.append(averaged_outputs.cpu().numpy())

all_outputs = np.concatenate(all_outputs, axis=0)

# For kappa (ordinal labels), converting to an expected rating often improves agreement vs argmax.
probs = torch.softmax(torch.from_numpy(all_outputs), dim=1).numpy()
classes = np.arange(5, dtype=np.float32)[None, :]
exp_score = (probs * classes).sum(axis=1)
final_predictions = np.rint(exp_score).astype(int)
final_predictions = np.clip(final_predictions, 0, 4)

print("Pred shape:", final_predictions.shape)



Inference: 100%|██████████| 23/23 [00:17<00:00,  1.34it/s]

Pred shape: (367,)


In [8]:
submission_df = pd.DataFrame(
    {
        "id_code": pd.read_csv(test_csv_file)["id_code"].values,
        "diagnosis": final_predictions.astype(int),
    }
)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print("Wrote:", submission_path)
print(submission_df.head())

Wrote: submission.csv
        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          2
3  6cbc3dad809c          2
4  a9bc2f892cb3          0
